# Sahit Bolla and Sahil Chute

#### Enchancing Scene Text Recognition Accuracy through Noise-Resilient Architectures

Imports

In [ ]:
from pathlib import Path
import re
import numpy as np

Setup

In [12]:
np.random.seed(42)

def tt_build_image_details(image_directory: str, annotation_directory: str):
    image_details = []

    for image_path in sorted(Path(image_directory).glob("*.jpg")):
        annotation_path = Path(annotation_directory) / ("poly_gt_" + image_path.stem + ".txt")
        image_details.append((str(image_path), str(annotation_path)))
    
    return image_details

tt_train_details = tt_build_image_details("Total-Text/Train", "Total-Text/Annotation/groundtruth_polygonal_annotation/Train")
tt_test_details_full = tt_build_image_details("Total-Text/Test", "Total-Text/Annotation/groundtruth_polygonal_annotation/Test")

print("Total-Text Train records:", len(tt_train_details))
print("Total-Text Test records:", len(tt_test_details_full))
print()

def tt_split_test_val(image_details):
    new_test_details = []
    new_val_details = []

    for index, record in enumerate(image_details):
        if index % 2 == 0:
            new_test_details.append(record)
        else:
            new_val_details.append(record)

    return new_test_details, new_val_details

tt_test_details, tt_val_details = tt_split_test_val(tt_test_details_full)
print("Test records after split:", len(tt_test_details))
print("Validation records after split:", len(tt_val_details))
print()

def parse_tt_annotation(annotation_path: Path):
    entries = []

    for line in annotation_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue

        x_text = re.search(r"x:\s*\[\[(.*?)\]\]", line)
        y_text = re.search(r"y:\s*\[\[(.*?)\]\]", line)
        ornt_text = re.search(r"ornt:\s*\[u?'([^']*)'\]", line)
        transcription_text = re.search(r"transcriptions:\s*\[u?'(.*)'\]\s*$", line)

        if not (x_text and y_text and transcription_text and ornt_text):
            continue

        x_coords = [int(coord) for coord in x_text.group(1).split()]
        y_coords = [int(coord) for coord in y_text.group(1).split()]
        ornt = ornt_text.group(1)
        transcription = transcription_text.group(1)

        if (len(x_coords) != len(y_coords)
            or len(x_coords) < 3
            or len(y_coords) < 3
            or ornt.strip() == "#"
            or transcription.strip() == ""
            or transcription.strip() == "#"):
            continue

        polygon = np.stack([x_coords, y_coords], axis=1)
        entries.append((transcription, polygon))

    return entries
    
tt_train_annotations = {Path(image_path).stem: parse_tt_annotation(Path(annotation_path)) for image_path, annotation_path in tt_train_details}
tt_val_annotations = {Path(image_path).stem: parse_tt_annotation(Path(annotation_path)) for image_path, annotation_path in tt_val_details}
tt_test_annotations = {Path(image_path).stem: parse_tt_annotation(Path(annotation_path)) for image_path, annotation_path in tt_test_details}

tt_total_train_polygons = sum(len(ann) for ann in tt_train_annotations.values())
tt_total_val_polygons = sum(len(ann) for ann in tt_val_annotations.values())
tt_total_test_polygons = sum(len(ann) for ann in tt_test_annotations.values())

print("Total polygons in train set:", tt_total_train_polygons)
print("Total polygons in validation set:", tt_total_val_polygons)
print("Total polygons in test set:", tt_total_test_polygons)


Total-Text Train records: 1255
Total-Text Test records: 300

Test records after split: 150
Validation records after split: 150

Total polygons in train set: 9277
Total polygons in validation set: 1060
Total polygons in test set: 1115
